# WoundSense: model training and external validation

This notebook walks through the pipeline the app uses. Run it from the project root.

1. **Training data** (`data/generate_training_data.py`): no public dataset records our signals from a wound dressing with healthy / healing / infected labels, so the training set is *knowledge-based*. Each class is sampled from distributions set by published criteria (SIRS, Sepsis-3/qSOFA, moist wound healing, hypertension and healing). Every source is a PubMed record in `data/literature_data.py`.
2. **Training** (`models/train_model.py`): a Random Forest classifier (condition) and a Random Forest regressor (days to heal) on **temperature, humidity, heart rate, systolic BP and diastolic BP**.
3. **External test** (`validation/validate_physionet2019.py`): the official **PhysioNet/CinC Challenge 2019** dataset, 40,336 real ICU patients from two hospital systems with Sepsis-3 labels. It is **used only for testing, never for training**.

In [ ]:
import json, subprocess, sys
import pandas as pd
from IPython.display import Image, display
PY = sys.executable

## 1. Generate the knowledge-based training set

In [ ]:
print(subprocess.run([PY, "data/generate_training_data.py"], capture_output=True, text=True).stdout)
train = pd.read_csv("data/wound_data_train.csv")
train.groupby("condition").describe().T.round(1).head(30)

## 2. Train the Random Forest classifier and regressor

In [ ]:
print(subprocess.run([PY, "models/train_model.py"], capture_output=True, text=True).stdout)
meta = json.load(open("models/metadata.json"))
pd.DataFrame(meta["condition_model"]["per_class"])

In [ ]:
pd.DataFrame({
    "classifier importance": meta["condition_model"]["feature_importance"],
    "regressor importance": meta["healing_model"]["feature_importance"],
})

## 3. External validation on PhysioNet 2019 (test only)

Download once (about 40k small files from PhysioNet's official AWS mirror), then evaluate.

* **Label:** `SepsisLabel` (Sepsis-3, positive from 6 h before onset). Sepsis is *systemic* infection, so it is the closest real, officially labelled proxy for our `infected` class. It is not local wound infection.
* **Missing input:** humidity is not recorded in ICU data, so the deployed model's output is averaged over the training humidity distribution. A vitals-only Random Forest is reported alongside.
* **Baselines on the same rows:** the SIRS vital-sign criteria and the qSOFA systolic-BP criterion.

In [ ]:
# subprocess.run([PY, "validation/download_physionet2019.py"])   # once, ~10 min
print(subprocess.run([PY, "validation/validate_physionet2019.py"], capture_output=True, text=True).stdout)
ev = json.load(open("models/external_validation.json"))
{k: v for k, v in ev["dataset"].items() if k not in ("preprocessing",)}

In [ ]:
cols = ["auroc", "auprc", "sensitivity", "specificity", "balanced_accuracy", "f1", "n"]
print("Patient level (highest risk during the ICU stay vs ever-septic)")
display(pd.DataFrame(ev["classifier_patient"]).T[cols])
print("Hourly rows")
display(pd.DataFrame(ev["classifier_hourly"]).T[cols])
print("Per hospital system:", ev["classifier_per_hospital"])

In [ ]:
for f in ev["figures"]:
    display(Image(filename=f"model_outputs/{f}"))

## 4. Regressor (days to heal)

No dataset records wound healing time, so the regressor cannot be scored for accuracy on real outcomes. It is checked for **construct validity** instead: predicted recovery time should be longer during sepsis and should rise as onset approaches.

In [ ]:
ev["regressor"]